In [1]:
import sympy as sp
import numpy as np
from scipy.optimize import fsolve
import sympy.physics.mechanics as me 
sp.init_printing(use_latex="mathjax")
me.init_vprinting()
from IPython.display import display

In [2]:
N, B, U1, U2, L1, L2 = sp.symbols('N, B, U_1, U_2, L_1, L_2', cls=me.ReferenceFrame)
O, T, C1, C2, R1, R2 = sp.symbols('O, T, C_1, C_2, R_1, R_2', cls=me.Point)                                 # CoM points for the bodies
A1, A2, B1, B2, K1, K2, O1, O2 = sp.symbols('A_1, A_2, B_1, B_2, K_1, K_2, O_1, O_2', cls=me.Point)         # points for constraints / loop closure
alpha, beta = me.dynamicsymbols('alpha beta', real="True")
theta1, theta2 = me.dynamicsymbols('theta1:3', real="True")
psi1, psi2 = me.dynamicsymbols('psi1:3', real="True")
phi1, phi2 = me.dynamicsymbols('phi1:3', real="True")

ls, lc, lr, lf = sp.symbols('l_s, l_c, l_r, l_f')
m_t, m_c, m_r, g = sp.symbols('m_t, m_c, m_r, g')
I_B_Bo = me.inertia(B, 1, 1, 1)
I_U1_U1o = me.inertia(U1, 1, 1, 1)
I_L1_L1o = me.inertia(L1, 1, 1, 1)
I_U2_U2o = me.inertia(U2, 1, 1, 1)
I_L2_L2o = me.inertia(L2, 1, 1, 1)
t = me.dynamicsymbols._t

T1, T2, T3, T4, T5 = me.dynamicsymbols('T1 T_joint18 T3 T_joint19 T5')


In [3]:
u1, u2, u3, u4, u5, u6, u7, u8 = me.dynamicsymbols('u1:9')
q = sp.Matrix([theta1, theta2])
q_r = sp.Matrix([alpha, beta, psi1, psi2, phi1, phi2])
q_N = q.col_join(q_r)
u = sp.Matrix([u1, u2])
u_r = sp.Matrix([u3, u4, u5, u6, u7, u8])
u_N = u.col_join(u_r)

qdot_N = q_N.diff(t)
udot = u.diff(t)

q_N_zero = {q:0 for q in q_N}
u_r_zero = {u: 0 for u in u_r}
u_N_zero = {u: 0 for u in u_N}
qdot_N_zero = {qd: 0 for qd in qdot_N}
u_d_zero = {ud: 0 for ud in udot}

q, q_r, q_N, u, u_r, u_N, qdot_N, udot

⎛            ⎡θ₁⎤              ⎡u₁⎤  ⎡θ₁̇⎤      ⎞
⎜            ⎢  ⎥              ⎢  ⎥  ⎢  ⎥      ⎟
⎜      ⎡α ⎤  ⎢θ₂⎥        ⎡u₃⎤  ⎢u₂⎥  ⎢θ₂̇⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢  ⎥  ⎢  ⎥  ⎢  ⎥      ⎟
⎜      ⎢β ⎥  ⎢α ⎥        ⎢u₄⎥  ⎢u₃⎥  ⎢α̇ ⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢  ⎥  ⎢  ⎥  ⎢  ⎥      ⎟
⎜⎡θ₁⎤  ⎢ψ₁⎥  ⎢β ⎥  ⎡u₁⎤  ⎢u₅⎥  ⎢u₄⎥  ⎢β̇ ⎥  ⎡u₁̇⎤⎟
⎜⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥⎟
⎜⎣θ₂⎦  ⎢ψ₂⎥  ⎢ψ₁⎥  ⎣u₂⎦  ⎢u₆⎥  ⎢u₅⎥  ⎢ψ₁̇⎥  ⎣u₂̇⎦⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢  ⎥  ⎢  ⎥  ⎢  ⎥      ⎟
⎜      ⎢φ₁⎥  ⎢ψ₂⎥        ⎢u₇⎥  ⎢u₆⎥  ⎢ψ₂̇⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢  ⎥  ⎢  ⎥  ⎢  ⎥      ⎟
⎜      ⎣φ₂⎦  ⎢φ₁⎥        ⎣u₈⎦  ⎢u₇⎥  ⎢φ₁̇⎥      ⎟
⎜            ⎢  ⎥              ⎢  ⎥  ⎢  ⎥      ⎟
⎝            ⎣φ₂⎦              ⎣u₈⎦  ⎣φ₂̇⎦      ⎠

In [4]:
B.orient_body_fixed(N, (alpha, beta, 0), 'XYZ')
U1.orient_axis(B, B.y, theta1)
L1.orient_body_fixed(U1, (psi1, psi2, 0), 'XYZ')

U2.orient_axis(B, B.y, theta2)
L2.orient_body_fixed(U2, (phi1, phi2, 0), 'XYZ')

O1.set_pos(O, lc*N.x + ls/2*N.y - lf*N.z)
O2.set_pos(O, lc*N.x - ls/2*N.y - lf*N.z)

T.set_pos(O, (lr - lf)*B.z)

A1.set_pos(T, ls/2*B.y)
C1.set_pos(A1, lc/2*U1.x)
B1.set_pos(C1, lc/2*U1.x)
R1.set_pos(B1, -lr/2*L1.z)
K1.set_pos(R1, -lr/2*L1.z)

A2.set_pos(T, -ls/2*B.y)
C2.set_pos(A2, lc/2*U2.x)
B2.set_pos(C2, lc/2*U2.x)
R2.set_pos(B2, -lr/2*L2.z)
K2.set_pos(R2, -lr/2*L2.z)


In [5]:

B2.pos_from(O) , O2.pos_from(O)

In [6]:
# Position dependency

me.find_dynamicsymbols(T.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C1.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C2.pos_from(O), reference_frame=N), me.find_dynamicsymbols(R1.pos_from(O), reference_frame=N),me.find_dynamicsymbols(K1.pos_from(O), reference_frame=N), me.find_dynamicsymbols(R2.pos_from(O), reference_frame=N),  me.find_dynamicsymbols(K2.pos_from(O), reference_frame=N)

In [18]:
# loop closure equations

loop_closure1 = K1.pos_from(O) - O1.pos_from(O)
loop_closure2 = K2.pos_from(O) - O2.pos_from(O)
rod_constraint1 = B1.pos_from(O) - O1.pos_from(O)
rod_constraint2 = B2.pos_from(O) - O2.pos_from(O)


## Algebraic Holonomic constraint equations

In [19]:
# Holonomic constraints

fh = sp.Matrix([
    rod_constraint1.dot(rod_constraint1) - lr**2,
    rod_constraint2.dot(rod_constraint2) - lr**2,
    loop_closure1.dot(N.y) - 0,
    loop_closure2.dot(N.y) - 0,
    loop_closure1.dot(N.z) - 0,
    loop_closure2.dot(N.z) - 0,
    # loop_closure1.dot(N.x) - 0,
    # loop_closure2.dot(N.x) - 0,
    ])

fh = fh.applyfunc(lambda i: sp.simplify(sp.trigsimp(i)))

In [20]:
fh

⎡                                                                              ↪
⎢       2                    2                                                 ↪
⎢- 2⋅l_c ⋅cos(β + θ₁) + 2⋅l_c  - 2⋅l_c⋅l_f⋅sin(β + θ₁)⋅cos(α) + 2⋅l_c⋅l_f⋅sin( ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢       2                    2                                                 ↪
⎢- 2⋅l_c ⋅cos(β + θ₂) + 2⋅l_c  - 2⋅l_c⋅l_f⋅sin(β + θ₂)⋅cos(α) + 2⋅l_c⋅l_f⋅sin( ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                          l_c ↪
⎢                           

### Solving for theta's

In [21]:
me.find_dynamicsymbols(fh[0]), me.find_dynamicsymbols(fh[1])

In [22]:

eqs = sp.Matrix([fh[0], fh[1]])
p = sp.Matrix([lc, lf, lr, ls])
i = sp.Matrix([alpha, beta])
o = sp.Matrix([theta1, theta2])

p_vals = np.array([0.075, 0.025, 0.115, 0.08])
i_vals = np.deg2rad([0.0, 10.0])
o_guess = np.deg2rad([0.0, 0.0]) 

eval_fh = sp.lambdify((o, i, p), eqs)

theta1_, theta2_ = fsolve(
    lambda o, i, p: np.squeeze(eval_fh(o, i, p)),
    o_guess,
    args=(i_vals, p_vals),
    xtol=1.49012e-10)

np.rad2deg([theta1_, theta2_])


array([-10.22956697, -10.22956697])

### Solving for phi's

In [27]:
me.find_dynamicsymbols(fh[2]), me.find_dynamicsymbols(fh[4])

In [31]:

eqs = sp.Matrix([fh[2], fh[4]])
p = sp.Matrix([lc, lf, lr, ls])
i = sp.Matrix([alpha, beta, theta1])
o = sp.Matrix([psi1, psi2])

p_vals = np.array([0.075, 0.025, 0.115, 0.08])
i_vals = np.deg2rad([0.0, 10.0, -10.22956697])
o_guess = np.deg2rad([0.0, 0.0]) 

eval_fh = sp.lambdify((o, i, p), eqs)

psi1_, psi2_ = fsolve(
    lambda o, i, p: np.squeeze(eval_fh(o, i, p)),
    o_guess,
    args=(i_vals, p_vals),
    xtol=1.49012e-10)

np.rad2deg([psi1_, psi2_])

array([-8.43125483e-16, -7.58071352e+00])

### Solving for psi's

In [29]:
me.find_dynamicsymbols(fh[3]), me.find_dynamicsymbols(fh[5]),

In [32]:
eqs = sp.Matrix([fh[3], fh[5]])
p = sp.Matrix([lc, lf, lr, ls])
i = sp.Matrix([alpha, beta, theta2])
o = sp.Matrix([phi1, phi2])

p_vals = np.array([0.075, 0.025, 0.115, 0.08])
i_vals = np.deg2rad([0.0, 10.0, -10.22956697])
o_guess = np.deg2rad([0.0, 0.0]) 

eval_fh = sp.lambdify((o, i, p), eqs)

phi1_, phi2_ = fsolve(
    lambda o, i, p: np.squeeze(eval_fh(o, i, p)),
    o_guess,
    args=(i_vals, p_vals),
    xtol=1.49012e-10)

np.rad2deg([phi1_, phi2_])

array([-8.43125483e-16, -7.58071352e+00])